# Notebook 03 — ChromaDB: A Full Vector Store

---

## What ChromaDB Adds Over FAISS

FAISS is a vector search library. ChromaDB is a vector database.

The difference:

| FAISS                          | ChromaDB                               |
|--------------------------------|----------------------------------------|
| Stores only vectors            | Stores vectors + text + metadata       |
| No filtering                   | Filter by metadata before/after search |
| No persistence by default      | Built-in persistent storage to disk    |
| Manual text management         | Retrieves original text automatically  |
| Lower level, more control      | Higher level, easier to use            |

For most RAG applications, ChromaDB is the better choice because you almost always
want to retrieve the text, not just a vector index.

---

## Key Concepts in ChromaDB

**Client** — your connection to ChromaDB. Can be in-memory or persistent.

**Collection** — a named group of documents. Think of it as a table.
Each collection has its own embeddings, text, and metadata.

**Document** — a piece of text you add to a collection.

**Embedding** — the vector for that document. You can provide it yourself or
let ChromaDB generate it using its default embedding function.

**Metadata** — a dict of extra information attached to each document.
Examples: source file name, page number, date, category.

**ID** — a unique string identifier for each document. Required.

---

In [1]:
# Install if needed
# !pip install chromadb sentence-transformers numpy

In [2]:
import sys
sys.path.append('..')

import chromadb
from utils.embedder import embed, embed_batch
from utils.searcher import get_chroma_collection, chroma_add, chroma_query, print_chroma_results

d:\Naveena Natarajan\Tutor\Abhishek-Bangalore-Python\Abhishek_Implementation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Part 1 — Creating a Client and Collection

In [3]:
# In-memory client — data is lost when the notebook restarts
# Good for experimenting without writing to disk
client = chromadb.Client()

# Create a collection called 'sales_docs'
# get_or_create_collection is safe to call multiple times
collection = client.get_or_create_collection(name="sales_docs")

print("Collection name :", collection.name)
print("Document count  :", collection.count())

Collection name : sales_docs
Document count  : 0


---

## Part 2 — Adding Documents

We provide our own embeddings (from sentence-transformers) rather than using ChromaDB's default.
This gives us control over the embedding model and ensures consistency across notebooks.

In [4]:
documents = [
    "Handle price objections by demonstrating ROI to the prospect.",
    "When a client says it costs too much, reframe around value and savings.",
    "Always conduct discovery questions before presenting any solution.",
    "The best salespeople listen 70 percent of the time in every conversation.",
    "Our quarterly revenue showed a strong upward trend this year.",
    "Company income increased 15 percent compared to last year.",
    "Never discount without attaching a business reason to the offer.",
    "Follow up after a demo with a short, specific message — not a generic check-in.",
]

metadatas = [
    {"source": "sales_manual", "chapter": "objections", "page": 42},
    {"source": "sales_manual", "chapter": "objections", "page": 45},
    {"source": "sales_manual", "chapter": "discovery",  "page": 12},
    {"source": "sales_manual", "chapter": "closing",    "page": 78},
    {"source": "quarterly_report", "chapter": "financials", "page": 3},
    {"source": "quarterly_report", "chapter": "financials", "page": 4},
    {"source": "sales_manual", "chapter": "objections", "page": 50},
    {"source": "sales_manual", "chapter": "follow_up",  "page": 60},
]

ids = [f"doc_{i}" for i in range(len(documents))]

# Compute embeddings
embeddings = embed_batch(documents)

# Add to ChromaDB
chroma_add(
    collection=collection,
    ids=ids,
    texts=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents added:", collection.count())

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.32it/s]

Documents added: 8


---

## Part 3 — Basic Semantic Query

In [5]:
query = "how to handle price objections"
query_embedding = embed(query).tolist()

results = chroma_query(collection, query_embedding, n_results=3)

print("Query:", query)
print()
print_chroma_results(results)

Query: how to handle price objections

Result 1  (similarity: 0.4674)
  Metadata : {'chapter': 'objections', 'page': 42, 'source': 'sales_manual'}
  Text     : Handle price objections by demonstrating ROI to the prospect.

Result 2  (similarity: 0.0122)
  Metadata : {'chapter': 'objections', 'page': 45, 'source': 'sales_manual'}
  Text     : When a client says it costs too much, reframe around value and savings.

Result 3  (similarity: -0.1772)
  Metadata : {'chapter': 'objections', 'page': 50, 'source': 'sales_manual'}
  Text     : Never discount without attaching a business reason to the offer.



---

## Part 4 — Filtering by Metadata

One of ChromaDB's main advantages over raw FAISS is metadata filtering.
You can restrict search to a specific source file, chapter, or any other metadata field.

In [6]:
# Only search inside the 'objections' chapter
query = "how should I respond when cost comes up"
query_embedding = embed(query).tolist()

results = chroma_query(
    collection,
    query_embedding,
    n_results=3,
    filters={"chapter": "objections"}
)

print("Query (filtered to chapter=objections):", query)
print()
print_chroma_results(results)

Query (filtered to chapter=objections): how should I respond when cost comes up

Result 1  (similarity: 0.0902)
  Metadata : {'chapter': 'objections', 'page': 45, 'source': 'sales_manual'}
  Text     : When a client says it costs too much, reframe around value and savings.

Result 2  (similarity: 0.0591)
  Metadata : {'chapter': 'objections', 'page': 42, 'source': 'sales_manual'}
  Text     : Handle price objections by demonstrating ROI to the prospect.

Result 3  (similarity: -0.3574)
  Metadata : {'chapter': 'objections', 'page': 50, 'source': 'sales_manual'}
  Text     : Never discount without attaching a business reason to the offer.



In [7]:
# Only search inside the quarterly report
query_financial = "how did the business perform this year"
query_embedding = embed(query_financial).tolist()

results = chroma_query(
    collection,
    query_embedding,
    n_results=2,
    filters={"source": "quarterly_report"}
)

print("Query (filtered to source=quarterly_report):", query_financial)
print()
print_chroma_results(results)

Query (filtered to source=quarterly_report): how did the business perform this year

Result 1  (similarity: -0.1352)
  Metadata : {'chapter': 'financials', 'page': 3, 'source': 'quarterly_report'}
  Text     : Our quarterly revenue showed a strong upward trend this year.

Result 2  (similarity: -0.2746)
  Metadata : {'chapter': 'financials', 'page': 4, 'source': 'quarterly_report'}
  Text     : Company income increased 15 percent compared to last year.



---

## Part 5 — Updating and Deleting Documents

In [8]:
# Update an existing document (upsert: add if not exists, update if exists)
updated_text = "Handle price objections by focusing on long-term ROI and total cost of ownership."
updated_embedding = embed(updated_text).tolist()

collection.update(
    ids=["doc_0"],
    documents=[updated_text],
    embeddings=[updated_embedding],
    metadatas=[{"source": "sales_manual", "chapter": "objections", "page": 42, "version": 2}]
)

print("doc_0 updated.")

doc_0 updated.


In [9]:
# Delete a document by id
collection.delete(ids=["doc_5"])
print("doc_5 deleted.")
print("Documents remaining:", collection.count())

doc_5 deleted.
Documents remaining: 7


---

## Part 6 — Persistent Storage

With an in-memory client, all data is lost when the notebook restarts.
Use `PersistentClient` to write data to disk.

In [10]:
# Create a persistent client — data is saved to the given path
persistent_client = chromadb.PersistentClient(path="../data/chroma_db")

persistent_collection = persistent_client.get_or_create_collection(name="persistent_sales")

# Add the same documents
all_embeddings = embed_batch(documents)
chroma_add(
    collection=persistent_collection,
    ids=[f"p_{i}" for i in range(len(documents))],
    texts=documents,
    embeddings=all_embeddings.tolist(),
    metadatas=metadatas
)

print("Saved to disk. Count:", persistent_collection.count())
print("Data path: ../data/chroma_db/")

Batches: 100%|██████████| 1/1 [00:00<00:00, 26.41it/s]
Insert of existing embedding ID: p_0
Insert of existing embedding ID: p_1
Insert of existing embedding ID: p_2
Insert of existing embedding ID: p_3
Insert of existing embedding ID: p_4
Insert of existing embedding ID: p_5
Insert of existing embedding ID: p_6
Insert of existing embedding ID: p_7
Add of existing embedding ID: p_0
Add of existing embedding ID: p_1
Add of existing embedding ID: p_2
Add of existing embedding ID: p_3
Add of existing embedding ID: p_4
Add of existing embedding ID: p_5
Add of existing embedding ID: p_6
Add of existing embedding ID: p_7


Saved to disk. Count: 8
Data path: ../data/chroma_db/


In [11]:
# Reload in a new client — data persists
reload_client = chromadb.PersistentClient(path="../data/chroma_db")
reload_collection = reload_client.get_or_create_collection(name="persistent_sales")

print("Reloaded collection count:", reload_collection.count())
print("Data survived a client restart.")

Reloaded collection count: 8
Data survived a client restart.


---

## Summary

- ChromaDB stores text, embeddings, and metadata together in one place.
- You can filter queries by metadata before similarity search runs.
- In-memory client is for quick experiments. PersistentClient writes to disk.
- ChromaDB handles the bookkeeping that you would have to manage manually with FAISS.

Next: Notebook 04 covers chunking — how to split long documents before embedding them.